In [1]:
import os

import lightning as L
import numpy as np
import torch
from torch.utils.data import DataLoader, TensorDataset

from deep_uncertainty.models.lightning_regressors import DoublePoissonNN
from deep_uncertainty.utils.generic_utils import get_yaml

In [2]:
config = get_yaml("../toy/toy_exp_train_config.yaml")

In [3]:
data = np.load(os.path.join("..", config['dataset']["dir"].replace(".", ".."), config['dataset']["name"] + ".npz"))
X_train, y_train = data["X_train"], data["y_train"]
X_val, y_val = data["X_val"], data["y_val"]
X_test, y_test = data["X_test"], data["y_test"]

In [4]:
train_dataset = TensorDataset(
    torch.Tensor(X_train.reshape(-1, 1)),
    torch.Tensor(y_train.reshape(-1, 1))
)
val_dataset = TensorDataset(
    torch.Tensor(X_val.reshape(-1, 1)),
    torch.Tensor(y_val.reshape(-1, 1))
)
test_dataset = TensorDataset(
    torch.Tensor(X_test.reshape(-1, 1)),
    torch.Tensor(y_test.reshape(-1, 1))
)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=9, persistent_workers=True)
val_loader = DataLoader(val_dataset, batch_size=1, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=1, shuffle=False)

In [14]:
model = DoublePoissonNN(input_dim=1, beta=0.5)
trainer = L.Trainer(min_epochs=1000, max_epochs=1000, log_every_n_steps=25, enable_checkpointing=False, enable_model_summary=False, accelerator="cpu")
trainer.fit(model=model, train_dataloaders=train_loader)

GPU available: True (mps), used: False
TPU available: False, using: 0 TPU cores
IPU available: False, using: 0 IPUs
HPU available: False, using: 0 HPUs
/Users/spenceryoung/miniconda3/envs/deep-uncertainty/lib/python3.10/site-packages/lightning/pytorch/trainer/setup.py:187: GPU available but not used. You can set it by doing `Trainer(accelerator='gpu')`.


Training: |          | 0/? [00:00<?, ?it/s]

`Trainer.fit` stopped: `max_epochs=1000` reached.


In [15]:
test_predictions = torch.cat(trainer.predict(model=model, dataloaders=test_loader, return_predictions=True))
mu_hat, phi_hat = [x.squeeze() for x in torch.split(test_predictions, [1, 1], dim=-1)]
mu_hat.shape, phi_hat.shape

/Users/spenceryoung/miniconda3/envs/deep-uncertainty/lib/python3.10/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:441: The 'predict_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=9` in the `DataLoader` to improve performance.


Predicting: |          | 0/? [00:00<?, ?it/s]

(torch.Size([100]), torch.Size([100]))

In [16]:
np.mean((y_test - mu_hat.detach().numpy())**2)

5.667702475874207